# Tidal Analysis — Python

Python translation of `spectrumCB_demo.m` and `run_spectrum_demo.m`.

The `spectrumCB` function computes a power spectral density (PSD) using **50% overlapping segments** (Welch's method) with linear detrending. No windowing is applied, matching the MATLAB original.

**Algorithm summary:**
1. Split the record into overlapping chunks of length `chunk`, stepping by `chunk/2`
2. Detrend each chunk
3. FFT → one-sided PSD, normalized per the MATLAB convention
4. Average PSDs across all chunks
5. Check Parseval's theorem: `∫ PSD df / var(data) ≈ 1`

## `spectrumCB` function

In [2]:
import numpy as np
from scipy.signal import detrend as scipy_detrend


def spectrumCB(time, data, chunk):
    """
    Compute a power spectral density using 50% overlapping segments.
    Mirrors the MATLAB spectrumCB_demo function.

    Parameters
    ----------
    time  : array, time in days
    data  : array, data values (must be NaN-free)
    chunk : int, number of samples per segment

    Returns
    -------
    f       : frequency vector [cpd]
    a       : averaged power spectral density [units² / cpd]
    parseval: ratio of spectral integral to data variance (should be ≈ 1)
    """
    data = np.asarray(data, dtype=float)
    time = np.asarray(time, dtype=float)
    chunk = int(chunk)

    # --- split into 50% overlapping chunks ---
    segments = []
    step = chunk // 2
    ind = 0
    while ind + chunk <= len(data):
        segments.append(data[ind : ind + chunk])
        ind += step

    # --- frequency vector ---
    dt = np.nanmean(np.diff(time))      # mean sample interval [days]
    fn = 1.0 / (2.0 * dt)               # Nyquist frequency [cpd]
    N  = chunk
    T  = dt * N                         # segment length [days]
    df = 1.0 / T                        # frequency resolution [cpd]
    f  = np.arange(0, fn + df / 2, df)  # one-sided frequency vector [cpd]
    nf = len(f)                         # N//2 + 1

    # --- compute and average PSD across segments ---
    A = np.empty((len(segments), nf))
    for i, seg in enumerate(segments):
        seg_dt   = scipy_detrend(seg)           # remove linear trend
        fft_vals = np.fft.fft(seg_dt)           # complex FFT
        amp = np.abs(fft_vals[:nf]) ** 2        # one-sided amplitude² (0…Nyquist)
        amp = amp / N ** 2                      # MATLAB normalization
        amp = amp * 2                           # recover variance in negative freqs
        amp = amp / df                          # PSD [units² / cpd]
        A[i] = amp

    a = A.mean(axis=0)                          # averaged PSD

    # --- Parseval check ---
    variance = np.nanstd(data) ** 2
    int_spec  = np.trapezoid(a, f)
    parseval  = int_spec / variance

    print(f"Segments used: {len(segments)}")
    print(f"Parseval check: ∫PSD df / var(data) = {parseval:.4f}  (ideal = 1.00)")
    return f, a, parseval


---
## Demo 1 — `sample_data.mat`

Mirrors `run_spectrum_demo.m`. The dataset is a ~333-day velocity record sampled at 30 min (dt = 0.5 hr = 1/48 day).

Reference lines:
- 1 cpd — diurnal tidal band
- 24/12.42 ≈ 1.93 cpd — semidiurnal M₂ tide
- 24/12.00 = 2.00 cpd — semidiurnal S₂ tide
- Local inertial / Coriolis frequency

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# --- load data ---
with open('sample_data.json') as f:
    sd = json.load(f)

data = np.array(sd['data'])
time = np.array(sd['time'])   # MATLAB serial date numbers [days]

# --- compute spectrum ---
num_chunks = 10
chunk_len  = len(data) // num_chunks

f, psd, parseval = spectrumCB(time, data, chunk_len)

# --- plot ---
fig, ax = plt.subplots(figsize=(9, 5))

ax.loglog(f[1:], psd[1:], color='steelblue', lw=1.2, label='PSD')

ref_lines = [
    (1,            'K₁  (1 cpd)',      'orange'),
    (24 / 12.42,   'M₂  (1.93 cpd)',   'crimson'),
    (24 / 12.00,   'S₂  (2.00 cpd)',   'firebrick'),
    (24 / 15.566,  'f_Cor (La Jolla)', 'seagreen'),
]
for freq, label, color in ref_lines:
    ax.axvline(freq, color=color, ls='--', lw=1, label=label)

ax.set_ylim([1e-6, 1e-1])
ax.set_xlabel('Frequency [cpd]')
ax.set_ylabel(r'$\Phi_v$ [(m/s)² / cpd]')
ax.set_title('Sample Spectrum — spectrumCB Python')
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()


---
## Demo 2 — La Jolla tide gauge

`la_jolla_tide.json` contains the **full** NOAA La Jolla tide gauge record (~100 years, hourly, in mm). Missing values are stored as `null` and are linearly interpolated before spectral analysis.


In [ ]:
# --- load La Jolla tide data ---
with open('la_jolla_tide.json') as f:
    lj = json.load(f)

sea_level = np.array([np.nan if v is None else v for v in lj['sea_level']])  # [mm]
dt_lj     = lj['metadata']['dt_hours'] / 24.0   # hours → days
time_lj   = np.arange(len(sea_level)) * dt_lj   # synthetic time axis [days]

print(f"Total samples:   {len(sea_level):,}  ({lj['metadata']['duration_years']} years)")
print(f"Missing (null):  {lj['metadata']['n_nan']:,}  ({lj['metadata']['n_nan']/len(sea_level)*100:.1f}%)")

# --- linearly interpolate missing values ---
nan_mask = np.isnan(sea_level)
idx = np.arange(len(sea_level))
sea_level_interp = np.interp(idx, idx[~nan_mask], sea_level[~nan_mask])

print(f"Sea level range: {sea_level_interp.min():.0f} – {sea_level_interp.max():.0f} mm (after interpolation)")


In [ ]:
# --- compute tidal spectrum ---
num_chunks_lj = 10
chunk_lj      = len(sea_level_interp) // num_chunks_lj

f_lj, psd_lj, parseval_lj = spectrumCB(time_lj, sea_level_interp, chunk_lj)


In [ ]:
# --- tidal spectrum plot ---
fig, ax = plt.subplots(figsize=(10, 5))

ax.loglog(f_lj[1:], psd_lj[1:], color='navy', lw=1.2, label='PSD')

# Reference tidal constituents
tidal_lines = [
    (24 / 25.82,  'O₁  (0.93 cpd)',    'orange'),
    (1.0,         'K₁  (1.00 cpd)',    'darkorange'),
    (24 / 12.42,  'M₂  (1.93 cpd)',    'crimson'),
    (24 / 12.00,  'S₂  (2.00 cpd)',    'firebrick'),
    (24 / 6.21,   'M₄  (3.87 cpd)',    'purple'),
]
for freq, label, color in tidal_lines:
    ax.axvline(freq, color=color, ls='--', lw=1, label=label)

ax.set_xlabel('Frequency [cpd]')
ax.set_ylabel(r'$\Phi_\eta$ [mm² / cpd]')
ax.set_title('La Jolla Tide Gauge — Power Spectral Density')
ax.legend(fontsize=8, loc='lower left')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Quick look — raw tidal signal

Plot a 30-day window to visually confirm the tidal variability before computing the spectrum.

In [ ]:
# Show ~30 days of data
nplot = int(30 / dt_lj)

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(time_lj[:nplot], sea_level_interp[:nplot], lw=0.8, color='navy')
ax.set_xlabel('Time [days from record start]')
ax.set_ylabel('Sea level [mm]')
ax.set_title('La Jolla Tide Gauge — first 30 days (interpolated)')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
